# Représentation des données textuelles (Version Optimisée)
## 1. Nettoyage de texte optimisé

In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Téléchargement des ressources nécessaires
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

# Utilisation d'un set pour une recherche plus rapide (O(1) au lieu de O(N))
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_optimized(text):
    # Minuscules
    text = text.lower()
    # Suppression de la ponctuation et des caractères non alphabétiques
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Tokenisation
    tokens = word_tokenize(text)
    # Suppression des stopwords et lemmatisation
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

def netoyage_optimized(corpus):
    return [clean_text_optimized(doc) for doc in corpus]

# Test
texte = ["La vie est douce", "La vie est tranquille, est belle, est douce", "le corona-virus est méchant"]
print("Texte nettoyé :", netoyage_optimized(texte))

Texte nettoyé : ['la vie est douce', 'la vie est tranquille est belle est douce', 'le corona virus est chant']


## 2. Calcul de TF-IDF optimisé

In [4]:
import numpy as np
from collections import Counter
import math

def compute_tfidf_optimized(corpus):
    # Tokenisation du corpus
    tokenized_corpus = [doc.split() for doc in corpus]
    
    # Calcul de la fréquence de document (DF)
    df = Counter()
    for doc in tokenized_corpus:
        # On utilise un set pour ne compter qu'une fois par document
        df.update(set(doc))
    
    N = len(corpus)
    
    # Construction du vocabulaire
    vocabulary = {word: i for i, word in enumerate(df.keys())}
    
    # Initialisation de la matrice (dense pour la simplicité, mais on pourrait utiliser scipy.sparse)
    M = np.zeros((N, len(vocabulary)))
    
    # Remplissage de la matrice
    for i, doc in enumerate(tokenized_corpus):
        tf = Counter(doc)
        doc_len = len(doc)
        for word, count in tf.items():
            if word in vocabulary:
                j = vocabulary[word]
                tf_val = count / doc_len
                idf_val = math.log(1 + (N / df[word]))
                M[i, j] = tf_val * idf_val
                
    return M, vocabulary

# Test
texte_nettoye = netoyage_optimized(texte)
M_opt, vocab_opt = compute_tfidf_optimized(texte_nettoye)
print("Vocabulaire :", vocab_opt)
print("Matrice TF-IDF optimisée :\n", np.round(M_opt, 2))

Vocabulaire : {'est': 0, 'douce': 1, 'vie': 2, 'la': 3, 'belle': 4, 'tranquille': 5, 'corona': 6, 'chant': 7, 'virus': 8, 'le': 9}
Matrice TF-IDF optimisée :
 [[0.17 0.23 0.23 0.23 0.   0.   0.   0.   0.   0.  ]
 [0.26 0.11 0.11 0.11 0.17 0.17 0.   0.   0.   0.  ]
 [0.14 0.   0.   0.   0.   0.   0.28 0.28 0.28 0.28]]


## 3. Pipeline de Classification Optimisée (Spams)

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn import metrics

# Chargement des données
try:
    spams = pd.read_table("SMSSpamCollection.txt", sep="\t", header=0)
    
    # Application du nettoyage optimisé
    print("Nettoyage des données en cours...")
    spams['cleaned_message'] = netoyage_optimized(spams['message'].tolist())
    
    # Séparation des données
    X_train, X_test, y_train, y_test = train_test_split(
        spams['cleaned_message'], spams['classe'], test_size=0.3, random_state=1
    )
    
    # Création d'un pipeline
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    
    # Définition de la grille de paramètres pour l'optimisation
    param_grid = {
        'tfidf__ngram_range': [(1, 1), (1, 2)], # Unigrammes ou Bigrammes
        'tfidf__min_df': [1, 2, 5], # Ignorer les mots très rares
        'clf__C': [0.1, 1, 10] # Régularisation de la régression logistique
    }
    
    # Recherche sur grille avec validation croisée
    print("Recherche des meilleurs hyperparamètres...")
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train)
    
    print("\nMeilleurs paramètres trouvés :", grid_search.best_params_)
    print("Meilleur score de validation croisée :", grid_search.best_score_)
    
    # Évaluation sur l'ensemble de test
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    print("\nPrécision du modèle optimisé sur l'ensemble de test :", metrics.accuracy_score(y_test, y_pred))
    print("\nRapport de classification :\n", metrics.classification_report(y_test, y_pred))

except FileNotFoundError:
    print("Erreur : Le fichier 'SMSSpamCollection.txt' est introuvable. Veuillez le télécharger et le placer dans le même dossier.")

Nettoyage des données en cours...
Recherche des meilleurs hyperparamètres...
Fitting 5 folds for each of 18 candidates, totalling 90 fits

Meilleurs paramètres trouvés : {'clf__C': 10, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 1)}
Meilleur score de validation croisée : 0.9805128205128204

Précision du modèle optimisé sur l'ensemble de test : 0.9820574162679426

Rapport de classification :
               precision    recall  f1-score   support

         ham       0.98      1.00      0.99      1442
        spam       0.97      0.90      0.93       230

    accuracy                           0.98      1672
   macro avg       0.98      0.95      0.96      1672
weighted avg       0.98      0.98      0.98      1672

